# Phase 3B-lite — repeated target-split replication

Run this in VS Code with a connected **Colab GPU** kernel, after Phase 3A completed in the same Drive checkout. It evaluates four new, pre-registered TEM-1 position splits and combines them with the already completed Phase 3A split.

This is the final experimental run for the project. The notebook locks all new splits before opening their test labels. After its matrix summary exists, do **not** run the launch cell again.

## 1. Confirm the GPU runtime

The output must show `CUDA available: True` and an NVIDIA GPU. If it does not, reconnect this notebook to **Colab → New Colab Server → GPU**.

In [ ]:
import platform
import subprocess
import sys

import torch

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. Reconnect this notebook to a Colab GPU first.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
subprocess.run(['nvidia-smi'], check=True)

## 2. Mount Drive and update the existing Phase 3A checkout

Phase 3B-lite must use the same `MyDrive/esm2-phase3a/` folder as Phase 3A because it reuses the completed Phase 3A summary and cached zero-shot predictions.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = Path('/content/drive/MyDrive/esm2-phase3a')
if not (REPO_DIR / '.git').is_dir():
    raise FileNotFoundError('The completed Phase 3A Drive checkout is missing. Do not create a fresh folder.')
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
print(subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip())
print(REPO_DIR)

## 3. Install the updated project

This attaches the pulled project version to this fresh Colab kernel.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/run_phase3b_lite.py', '--help'], cwd=REPO_DIR, check=True)
print('Setup complete.')

## 4. Create and inspect the protocol lock

This is safe: it creates four position-disjoint manifests for split seeds 143–146 but does not run a model or evaluate a new test set. Read the printed output and make sure it reports four new split seeds and `test_evaluated: false`.

In [ ]:
subprocess.run([sys.executable, 'scripts/lock_phase3b_lite_protocol.py'], cwd=REPO_DIR, check=True)

## 5. Run the locked four-split matrix

The command reuses target-independent pooled/Reptile initializations across the four new splits, then creates the combined five-split analysis. If Colab disconnects, reconnect and rerun this cell: completed splits are skipped. If the run says that a split has predictions but no summary, stop and save the error rather than trying to rerun it.

In [ ]:
RUN_ROOT = REPO_DIR / 'results' / 'phase3b_lite'
if (RUN_ROOT / 'matrix_summary.json').exists():
    raise RuntimeError('The Phase 3B-lite matrix is already complete. Use the next cell to inspect it; do not rerun.')

subprocess.run(
    [sys.executable, 'scripts/run_phase3b_lite.py', '--device', 'cuda', '--microbatch-size', '16'],
    cwd=REPO_DIR,
    check=True,
)

## 6. Inspect the five-split replication

The primary result is `pooled_minus_standard`: its mean, spread, and win/tie/loss count across the five target partitions.

In [ ]:
import json
import pandas as pd

summary_path = RUN_ROOT / 'analysis' / 'summary.json'
if not summary_path.exists():
    raise FileNotFoundError('No complete Phase 3B-lite analysis yet. Finish the launch cell first.')
summary = json.loads(summary_path.read_text())
display(pd.DataFrame([summary['primary_comparison'], *summary['secondary_comparisons']]))
display(pd.read_csv(RUN_ROOT / 'analysis' / 'per_split_metrics.csv'))
print(f'Saved analysis: {summary_path}')